# ClientCo — Test des deux labels, liste mixte et répartition de K

**Objectif** : sur les clients réguliers, le churn partiel (plafond de CA plus grand) vaut-il le churn total (mieux prédit) ? Test avec un même modèle (HistGradientBoosting, 12 variables), marge nette à rétention 10 / 20 / 30 %, fusion des labels et répartition de K entre les deux modèles. Photo de test : 22/03/2024 (dates d'origine). Les décisions retenues sont dans `docs/modelisation.md`.

> **Capacité (25/11).** Hypothèse retenue : 500 appels par semaine pour l'entreprise, soit **K ≈ 700** par liste mensuelle à l'échelle du dataset. Les tests de ce notebook utilisent K = 1 000 et K = 4 000 (liste mensuelle et six listes cumulées) : même conclusion (10 à 25 % des places au total) ; K = 1 000 est un arrondi haut de 600 à 760 appels par mois.

Jour d'achat = CA net des retours du jour > 0,01 €.

## 0. Setup

Le CSV fait **4,2 Go / 63 M de lignes** : impossible à charger dans pandas sur un laptop. On utilise **DuckDB** : conversion unique en Parquet (~1 min, colonnée et compressée), puis on agrège en SQL et on ne rapatrie dans pandas que de petites tables pour tracer.

In [1]:
from pathlib import Path
import time
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mtick
from matplotlib.colors import LinearSegmentedColormap

pd.set_option("display.max_columns", 50, "display.width", 200, "display.float_format", lambda x: f"{x:,.3f}")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV = next((p for p in (ROOT / "data" / "dataset_sales_2026.csv", ROOT / "dataset_sales_2026.csv") if p.exists()), ROOT / "dataset_sales_2026.csv")
PARQUET = ROOT / "data" / "sales.parquet"

con = duckdb.connect()
con.execute("SET memory_limit='4GB'")
(ROOT / "data").mkdir(exist_ok=True)
con.execute(f"SET temp_directory='{ROOT / 'data' / '.duckdb_tmp'}'")


def q(sql: str) -> pd.DataFrame:
    """Run a SQL query and return a pandas DataFrame."""
    return con.execute(sql).fetchdf()


if not PARQUET.exists():
    t0 = time.time()
    con.execute(f"""
        COPY (SELECT * FROM read_csv('{CSV}', delim=';', header=true, columns={{
            'date_order':'DATE','date_invoice':'DATE','product_id':'BIGINT','client_id':'BIGINT',
            'sales_net':'DOUBLE','quantity':'DOUBLE','order_channel':'VARCHAR','branch_id':'BIGINT'}}))
        TO '{PARQUET}' (FORMAT parquet, COMPRESSION zstd)""")
    print(f"CSV -> Parquet en {time.time() - t0:.0f}s")

con.execute(f"CREATE OR REPLACE VIEW sales AS SELECT * FROM read_parquet('{PARQUET}')")
q("SELECT * FROM sales LIMIT 5")

,date_order,date_invoice,product_id,client_id,sales_net,quantity,order_channel,branch_id
0,2022-09-25,2022-09-25,2376333,188502,155.443,3.000,online,2732
1,2022-09-25,2022-09-25,2520527,835089,16.394,3.000,at the store,10279
2,2022-09-25,2022-09-25,2536756,1977896,365.766,21.000,online,6184
3,2022-09-25,2022-09-25,3028673,598816,8.599,201.000,at the store,4200
4,2022-09-25,2022-09-25,203377,2086861,"1,093.374",3.000,by phone,7051


### Style graphique
Palette catégorielle validée (ordre fixe : une couleur = une entité), rampe bleue pour les magnitudes, gris neutres pour la structure, orange pour « ce qui alerte ». Fond clair pour que les figures restent lisibles quel que soit le thème de l'éditeur.

In [3]:
C = dict(blue="#2a78d6", orange="#eb6834", aqua="#1baf7a", yellow="#eda100", magenta="#e87ba4",
         green="#008300", violet="#6250d6", red="#e34948")
INK, INK2, MUTED, GRID, AXIS, SURF = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
SEQ = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
CMAP = LinearSegmentedColormap.from_list("seq_blue", SEQ)
NEUTRAL = "#b9b8b0"

# Une couleur par canal, identique dans tout le notebook
CHANNEL_COLOR = {"by phone": C["orange"], "at the store": C["blue"], "online": C["aqua"],
                 "during the visit of a sales rep": C["yellow"], "other": C["magenta"]}
CHANNEL_LABEL = {"by phone": "Téléphone", "at the store": "En agence", "online": "En ligne",
                 "during the visit of a sales rep": "Visite commercial", "other": "Autre"}

plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "font.family": "sans-serif", "font.sans-serif": ["Helvetica Neue", "Arial", "DejaVu Sans"], "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlecolor": INK,
    "lines.linewidth": 2, "figure.dpi": 110, "legend.frameon": False, "legend.fontsize": 9,
    "axes.labelsize": 9.5, "xtick.labelsize": 9, "ytick.labelsize": 9,
})


def headline(fig, title, sub=None):
    """Action title (the takeaway) + optional muted subtitle, left-aligned above the figure."""
    h = fig.get_figheight()
    fig.text(0.005, 1 + (0.30 if sub else 0.12) / h, title, fontsize=13, fontweight="bold", color=INK, ha="left", va="bottom")
    if sub:
        fig.text(0.005, 1 + 0.08 / h, sub, fontsize=9.5, color=INK2, ha="left", va="bottom")


def fmt_m(x, _=None):
    return f"{x / 1e6:,.0f}"


def pct_axis(ax, axis="y", decimals=0):
    f = mtick.PercentFormatter(1.0, decimals=decimals)
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(f)


def label_end(ax, x, y, text, dx=6, **kw):
    """Direct label at the end of a line (secondary ink, never the series colour)."""
    ax.annotate(text, (x, y), xytext=(dx, 0), textcoords="offset points", va="center", color=INK2, fontsize=9, **kw)


def pct(x, d=1):
    return f"{100 * x:.{d}f} %"

### Table (client, jour d'achat)

Construite une fois à partir du Parquet (une passe sur les 63 M de lignes), comme dans `01_data_quality_and_activity_overview`.

In [8]:
# Daily client x day table (one pass over the 63M lines), then purchase days only.
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE cd AS
SELECT client_id, date_order AS d,
       sum(sales_net)                                   AS rev_net,
       sum(greatest(sales_net, 0))                      AS rev_pos,
       sum(least(sales_net, 0))                         AS rev_neg,
       count(*)                                         AS n_lines,
       count(DISTINCT product_id)                       AS n_prod,
       arg_max(branch_id, greatest(sales_net, 0))       AS branch_id,
       arg_max(order_channel, greatest(sales_net, 0))   AS channel,
       avg(least(greatest(date_diff('day', date_order, date_invoice), 0), 365)) AS lag_mean
FROM sales GROUP BY client_id, date_order""")
con.execute("CREATE OR REPLACE TABLE bd AS SELECT * FROM cd WHERE rev_pos > 0")
cnt = q("SELECT (SELECT count(*) FROM cd) AS client_days, (SELECT count(*) FROM bd) AS purchase_days").iloc[0]
print(f"{cnt['client_days']:,.0f} (client, jour) · dont {cnt['purchase_days']:,.0f} jours d'achat · construit en {time.time() - t0:.0f}s")

7,955,289 (client, jour) · dont 7,953,336 jours d'achat · construit en 16s


### Paramètres et table des jours d'achat nets (repris de `04_churn_scope_total_vs_partial`)

In [ ]:
# --- Paramètres du cadrage (modifiables)
T_SCOPE = "2023-09-22"                    # photo unique : 365 j d'historique avant, 360 j de recul après (toutes les données)
H_SC = 180                                # largeur des fenêtres A, B et C
DROP, RISE = 0.5, 0.5                     # churn partiel : CA(B) < 50 % de CA(A) ; contre-poids : CA(B) > 150 %
MIN_PURCH = 6                             # écarts d'achat fiables à partir de 6 jours d'achat sur 365 j
K_INT, H_MIN, H_MAX = 3.0, 30, 180        # interruption : silence >= clip(k x écart p90 du client, 30, 180 j)
SEGS = ["Rapide (écart méd. ≤ 7 j)", "Moyen (8–30 j)", "Lent (> 30 j)", "Historique court (< 6 achats)"]

# Jour d'achat = CA net du jour > 1 centime (un achat retourné le jour même n'est pas un achat ; le seuil évite que le
# bruit de virgule flottante d'un net de ±1e-13 fasse basculer le résultat d'une exécution à l'autre) ; `rev` = ce CA net
con.execute("CREATE OR REPLACE TABLE bn AS SELECT client_id, d, rev_net AS rev FROM cd WHERE rev_net > 0.01")
DATA_END = pd.Timestamp(q("SELECT max(d) AS m FROM bn").iloc[0, 0])

In [38]:
# Saisonnalité 2/2 : les mêmes définitions sur 13 photos mensuelles (A = 180 j avant T, B = 180 j après)
rows = []
for T in pd.date_range("2023-03-22", "2024-03-22", freq=pd.DateOffset(months=1)):
    T = str(T.date())
    r = q(f"""
    WITH a AS (SELECT client_id, sum(rev) AS ra, count(*) AS na FROM bn WHERE d >= DATE '{T}' - {H_SC} AND d < DATE '{T}' GROUP BY 1),
    b AS (SELECT client_id, sum(rev) AS rb, count(*) AS nb FROM bn WHERE d >= DATE '{T}' AND d < DATE '{T}' + {H_SC} GROUP BY 1)
    SELECT a.ra, a.na, coalesce(b.rb, 0) AS rb, coalesce(b.nb, 0) AS nb FROM a LEFT JOIN b USING (client_id)""")
    row = {"Photo T": T + (" ◀ retenue" if T == T_SCOPE else ""), "Clients": f"{len(r):,.0f}"}
    for lab, g in [("tous", r), ("réguliers", r[r.na >= 15])]:
        tot = g.ra.sum()
        part = (g.nb > 0) & (g.rb < (1 - DROP) * g.ra)
        rise = (g.nb > 0) & (g.rb > (1 + RISE) * g.ra)
        row[f"Total, {lab}"] = g.ra[g.nb == 0].sum() / tot
        row[f"Partiel, {lab}"] = (g.ra - g.rb)[part].sum() / tot
        row[f"Hausse, {lab}"] = (g.rb - g.ra)[rise].sum() / tot
        row[f"B/A, {lab}"] = g.rb.sum() / tot
    rows.append(row)
season = pd.DataFrame(rows).set_index("Photo T")
spread = season.drop(columns="Clients").astype(float).agg(["min", "max"])
print("Parts du CA de référence de la base de chaque photo ; B/A = CA(B) / CA(A) de toute la base ; MIN et MAX : extrêmes des 13 photos.")
pd.concat([season.assign(**{c: season[c].map("{:.1%}".format) for c in season.columns if not c.startswith(("B/A", "Clients"))},
                         **{c: season[c].map("{:.2f}".format) for c in season.columns if c.startswith("B/A")}),
           spread.map(lambda v: f"{v:.1%}" if v < 0.5 else f"{v:.2f}").set_axis(["MIN", "MAX"]).reindex(columns=season.columns, fill_value="")])

Parts du CA de référence de la base de chaque photo ; B/A = CA(B) / CA(A) de toute la base ; MIN et MAX : extrêmes des 13 photos.


,Clients,"Total, tous","Partiel, tous","Hausse, tous","B/A, tous","Total, réguliers","Partiel, réguliers","Hausse, réguliers","B/A, réguliers"
2023-03-22,"119,738",2.6%,13.6%,13.6%,0.91,0.9%,11.1%,9.4%,0.91
2023-04-22,"118,627",2.9%,12.8%,14.4%,0.93,1.1%,10.6%,9.6%,0.91
2023-05-22,"115,733",2.7%,10.5%,21.1%,1.07,1.0%,8.3%,15.8%,1.05
2023-06-22,"115,948",2.6%,10.3%,20.5%,1.06,1.0%,8.2%,14.9%,1.04
2023-07-22,"116,767",2.7%,11.0%,18.5%,1.02,0.9%,8.8%,12.9%,1.00
2023-08-22,"117,917",2.5%,12.2%,16.1%,0.97,0.9%,10.0%,10.8%,0.96
2023-09-22 ◀ retenue,"118,009",2.8%,11.5%,18.7%,1.01,1.0%,9.3%,13.4%,1.00
2023-10-22,"118,381",2.9%,11.1%,20.6%,1.04,1.2%,8.7%,15.3%,1.03
2023-11-22,"120,479",3.2%,13.6%,13.7%,0.90,1.3%,11.2%,9.3%,0.89
2023-12-22,"121,311",3.1%,14.0%,12.4%,0.88,1.2%,11.7%,8.3%,0.87


**Lecture de la saisonnalité** (indice 100 = CA par jour moyen ; les taux des 13 photos sont des parts du CA de référence de la base de chaque photo).

- **L'activité est nettement saisonnière.** Le creux de mai se répète deux années de suite (71 en 2023, 75 en 2024) et le pic est en août (118 et 123), soit un écart d'environ 1,7 entre les deux. Septembre 2023 (76) est un creux sans équivalent exploitable, septembre 2022 étant incomplet. Avril varie fortement d'une année à l'autre (82 contre 98). La croissance d'une année sur l'autre est de +4,7 % sur les 11 mois comparables : avec seulement deux ans, on ne sépare pas proprement la saison de la tendance.
- **Le churn total est peu sensible à la date de la photo.** Entre 2,5 % et 3,2 % du CA selon la photo (0,8 à 1,3 % pour les réguliers), soit environ ±12 % autour de la moyenne. Le plafond de 125 M€ de la photo retenue vaut donc en réalité une fourchette d'environ 110 à 140 M€. Les maximums sont sur les photos de novembre-décembre, dont la fenêtre B couvre l'hiver et le creux de mai.
- **Le churn partiel et le contre-poids sont, eux, très saisonniers.** Le partiel varie de 10,3 % à 14,0 % du CA et les hausses de 12,4 % à 21,1 %, ce qui s'explique par le rapport CA(B) / CA(A) de toute la base, qui oscille de 0,88 à 1,07. Les photos qui précèdent un creux (novembre, décembre) gonflent le partiel ; les photos de mai à juin, qui précèdent l'été, gonflent les hausses. Comparer A et B revient à comparer deux saisons : **le label partiel mélange perte de client et calendrier**.
- **La photo retenue est neutre par chance** (B/A = 1,01) : ses chiffres sont au milieu des fourchettes. Ce n'était pas un choix délibéré.
- **Entraînement et test.** Les photos d'entraînement prévues (22/03 au 22/09/2023) donnent un churn total de 2,5 à 2,9 % ; la photo de test (22/03/2024), 2,5 %. Le taux de base reste proche d'un côté à l'autre du découpage pour le total. En revanche, aucune photo d'entraînement ne part de novembre-décembre, là où les taux montent : le modèle ne verra pas une dérive saisonnière et ses performances en production sur cette période ne sont pas mesurables avec ces données.

**Conséquences pour les décisions.** (1) Le churn total est le label le plus robuste à la saison : un argument de plus pour le prendre comme label principal et traiter le partiel en variante, avec cette réserve. (2) Pas de variable « mois » : un seul recoupement annuel, le modèle la mémoriserait. Préférer des variables relatives au rythme du client, éventuellement normalisées par l'activité globale du moment. (3) Les dates sont fictives et décalées : le « creux de mai » est un motif réel des données mais son explication (jours fériés) ne l'est pas forcément.

## 1. Test de label : churn total contre churn partiel brut, sur les clients réguliers

**Question Q9 ter.** Le churn partiel a un plafond de CA plus grand que le total (`04_churn_scope_total_vs_partial`), mais on le soupçonne plus bruité et plus saisonnier. Le recommander en variante reposait sur une hypothèse non testée : il serait plus difficile à prédire. On la teste, avec **le même modèle, les mêmes variables et la même mesure** ; seul le label change.

**Protocole** (celui du document de décisions, en version courte).
- **Population : les réguliers**, c'est-à-dire au moins 6 jours d'achat sur les 180 j avant la photo et un écart médian ≤ 30 j (rapides + moyens, recalculés sur 180 j car les premières photos n'ont pas 365 j d'historique).
- **Labels** (fenêtre B de 180 j après la photo). *Total* : aucun jour d'achat dans B. *Partiel brut* : au moins un achat dans B, mais CA(B) < 50 % du CA des 180 j précédents. Pas de version durable : elle ne se valide pas dans le temps (`04_churn_scope_total_vs_partial`).
- **Variables** (12 de l'ensemble « basic » du document de décisions, tirées des seuls 180 j avant T) : récence, récence rapportée à l'écart p90 et à l'écart médian, fréquence sur 180 / 90 / 90 précédents jours et sa tendance, CA sur les mêmes périodes et sa tendance, écarts médian et p90.
- **Découpage temporel strict avec embargo.** Entraînement : 7 photos mensuelles du 22/03/2023 au 22/09/2023 (leur fenêtre B finit le 20/03/2024). Test : photo du 22/03/2024, jamais utilisée pour régler quoi que ce soit.
- **Modèle** : arbres boostés (HistGradientBoosting, réglages par défaut), probabilités calibrées par régression isotonique sur des prédictions hors échantillon (plis groupés par client).
- **Score** : probabilité × V, V étant le CA des 180 j annualisé. **Mesure : part du CA réellement perdu captée dans les K premiers clients**, avec intervalle de confiance par bootstrap sur les clients. On y ajoute une baseline sans apprentissage (récence ÷ écart p90, × V).
- **Même K pour les deux labels** : une liste de K clients a le même coût d'appels quel que soit le label.

Les montants sont dans le périmètre du dataset (pas extrapolés au CA de 15,5 Md€) et la rétention de 20 % est une hypothèse du chiffrage d'équipe, identique pour les deux labels dans le tableau de décision.

In [39]:
# Test de label 1/3 : tables de photos (variables avant T, labels après T) pour les clients réguliers
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold

DU_LOOK, DU_H = 180, 180                     # historique des variables ; largeur de la fenêtre de label B
DU_TRAIN = [str(t.date()) for t in pd.date_range("2023-03-22", "2023-09-22", freq=pd.DateOffset(months=1))]
DU_TEST = "2024-03-22"
DU_FEATS = ["recency", "recency_ratio", "recency_med_ratio", "freq_180", "freq_90", "freq_prev_90", "freq_trend",
            "rev_180", "rev_90", "rev_prev_90", "rev_trend", "gap_median", "gap_p90"]


def du_photo(T: str) -> pd.DataFrame:
    """Réguliers actifs sur les 180 j avant T : variables (strictement avant T), labels (180 j après T), V."""
    df = q(f"""
    WITH w AS (SELECT client_id, d, rev, d - lag(d) OVER (PARTITION BY client_id ORDER BY d) AS gap
               FROM bn WHERE d < DATE '{T}' AND d >= DATE '{T}' - {DU_LOOK}),
    f AS (SELECT client_id, count(*) AS freq_180, count(*) FILTER (WHERE d >= DATE '{T}' - 90) AS freq_90,
                 count(*) FILTER (WHERE d < DATE '{T}' - 90) AS freq_prev_90,
                 sum(rev) AS rev_180, coalesce(sum(rev) FILTER (WHERE d >= DATE '{T}' - 90), 0) AS rev_90,
                 coalesce(sum(rev) FILTER (WHERE d < DATE '{T}' - 90), 0) AS rev_prev_90,
                 median(gap) AS gap_median, quantile_cont(gap, 0.9) AS gap_p90, max(d) AS last_d FROM w GROUP BY 1),
    y AS (SELECT client_id, coalesce(sum(rev) FILTER (WHERE d < DATE '{T}' + {DU_H}), 0) AS rev_b,
                 count(*) FILTER (WHERE d < DATE '{T}' + {DU_H}) AS n_b,
                 coalesce(sum(rev) FILTER (WHERE d >= DATE '{T}' + {DU_H}), 0) AS rev_c      -- fenêtre C, pour la persistance
          FROM bn WHERE d >= DATE '{T}' AND d < DATE '{T}' + 2 * {DU_H} GROUP BY 1)
    SELECT f.*, date_diff('day', f.last_d, DATE '{T}') AS recency, coalesce(y.rev_b, 0) AS rev_b, coalesce(y.n_b, 0) AS n_b,
           coalesce(y.rev_c, 0) AS rev_c
    FROM f LEFT JOIN y USING (client_id)
    WHERE f.freq_180 >= 6 AND f.gap_median <= 30""")
    df = df.sort_values("client_id").reset_index(drop=True)             # ordre fixe : résultats reproductibles
    df["T"] = T
    df["recency_ratio"] = df.recency / df.gap_p90.clip(lower=1)
    df["recency_med_ratio"] = df.recency / df.gap_median.clip(lower=1)
    df["freq_trend"] = df.freq_90 / (df.freq_prev_90 + 1)
    df["rev_trend"] = df.rev_90 / (df.rev_prev_90 + 1)
    df["V"] = 2 * df.rev_180                                            # CA des 180 j, annualisé
    df["y_total"] = (df.n_b == 0).astype(int)
    df["y_part"] = ((df.n_b > 0) & (df.rev_b < (1 - DROP) * df.rev_180)).astype(int)
    df["lost_total"] = df.V * df.y_total                                # CA annualisé perdu si le client part
    df["lost_part"] = 2 * (df.rev_180 - df.rev_b) * df.y_part           # CA annualisé perdu si la baisse persiste
    return df


assert pd.Timestamp(DU_TRAIN[-1]) + pd.Timedelta(days=DU_H) <= pd.Timestamp(DU_TEST), "embargo non respecté"
du_tr = pd.concat([du_photo(t) for t in DU_TRAIN], ignore_index=True)
du_te = du_photo(DU_TEST)
pop = pd.DataFrame({
    "Taux de base, entraînement": [du_tr.y_total.mean(), du_tr.y_part.mean()],
    "Taux de base, test": [du_te.y_total.mean(), du_te.y_part.mean()],
    "Événements, test": [du_te.y_total.sum(), du_te.y_part.sum()],
    "CA en jeu, test (M)": [du_te.lost_total.sum() / 1e6, du_te.lost_part.sum() / 1e6]}, index=["Churn total", "Churn partiel brut"])
print(f"Entraînement : {len(du_tr):,} lignes client x photo ({len(DU_TRAIN)} photos) · test : {len(du_te):,} clients réguliers au {DU_TEST}")
pop.assign(**{c: pop[c].map("{:.1%}".format) for c in pop.columns[:2]},
           **{"Événements, test": pop["Événements, test"].map("{:,.0f}".format), "CA en jeu, test (M)": pop["CA en jeu, test (M)"].map("{:,.0f}".format)})

Entraînement : 421,994 lignes client x photo (7 photos) · test : 61,286 clients réguliers au 2024-03-22


,"Taux de base, entraînement","Taux de base, test","Événements, test","CA en jeu, test (M)"
Churn total,3.7%,3.6%,"2,226",61
Churn partiel brut,22.2%,23.5%,"14,420",505


In [40]:
# Test de label 2/3 : un modèle par label (mêmes variables, mêmes réglages), probabilités calibrées, scores P x V
def du_fit_predict(ycol: str) -> np.ndarray:
    X, y = du_tr[DU_FEATS], du_tr[ycol]
    oof = np.zeros(len(du_tr))
    for a, b in GroupKFold(3).split(X, y, du_tr.client_id):                      # plis groupés par client
        oof[b] = HistGradientBoostingClassifier(random_state=0).fit(X.iloc[a], y.iloc[a]).predict_proba(X.iloc[b])[:, 1]
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(oof, y)
    final = HistGradientBoostingClassifier(random_state=0).fit(X, y)
    return iso.predict(final.predict_proba(du_te[DU_FEATS])[:, 1])


du_p = {"total": du_fit_predict("y_total"), "part": du_fit_predict("y_part")}
V = du_te.V.to_numpy()
du_scores = {"Taille seule : V (plus gros clients)": V,
             "Baseline : récence ÷ p90, × V": du_te.recency_ratio.to_numpy() * V,
             "Modèle total : P(total) × V": du_p["total"] * V,
             "Modèle partiel : P(partiel) × V": du_p["part"] * V}
metrics = pd.DataFrame({lab: {"ROC-AUC": roc_auc_score(du_te[f"y_{lab}"], du_p[lab]),
                              "PR-AUC": average_precision_score(du_te[f"y_{lab}"], du_p[lab]),
                              "Taux de base": du_te[f"y_{lab}"].mean()} for lab in du_p}).rename(
    columns={"total": "Modèle total", "part": "Modèle partiel"})
metrics.map("{:.3f}".format)

,Modèle total,Modèle partiel
ROC-AUC,0.893,0.675
PR-AUC,0.378,0.376
Taux de base,0.036,0.235


In [41]:
# Test de label 3/3 : CA réellement perdu capté dans les K premiers clients (même K pour les deux labels)
KS = [1000, 2000, 5000]
LOST = {"total": du_te.lost_total.to_numpy(), "part": du_te.lost_part.to_numpy()}
LAB = {"total": "Perte évaluée : churn total", "part": "Perte évaluée : churn partiel brut"}


def capture(score, lost, k, idx=None):
    if idx is not None:
        score, lost = score[idx], lost[idx]
    return lost[np.argpartition(-score, k)[:k]].sum() / lost.sum()


boot = np.random.default_rng(0)
du_idx = [boot.integers(0, len(du_te), len(du_te)) for _ in range(200)]          # rééchantillonnage des clients
rows = []
for lab in ("total", "part"):
    for name, s in du_scores.items():
        r = {"Perte évaluée": LAB[lab].split(" : ")[1], "Score": name}
        for k in KS:
            lo, hi = np.percentile([capture(s, LOST[lab], k, i) for i in du_idx], [2.5, 97.5])
            r[f"Top {k:,}"] = f"{capture(s, LOST[lab], k):.1%}  [{lo:.0%}–{hi:.0%}]"
        rows.append(r)
print("Part du CA perdu captée dans les K premiers clients (intervalle de confiance à 95 % entre crochets).")
cap_tab = pd.DataFrame(rows).set_index(["Perte évaluée", "Score"])
cap_tab

Part du CA perdu captée dans les K premiers clients (intervalle de confiance à 95 % entre crochets).


Top 1,000         Top 2,000         Top 5,000
Perte évaluée      Score                                                                                     
churn total        Taille seule : V (plus gros clients)    8.4%  [2%–17%]   14.5%  [7%–22%]  22.4%  [15%–30%]
                   Baseline : récence ÷ p90, × V         46.6%  [40%–52%]  56.6%  [52%–62%]  67.4%  [63%–72%]
                   Modèle total : P(total) × V           56.2%  [51%–62%]  65.7%  [61%–71%]  76.8%  [73%–81%]
                   Modèle partiel : P(partiel) × V       18.2%  [11%–27%]  25.9%  [18%–34%]  39.0%  [32%–46%]
churn partiel brut Taille seule : V (plus gros clients)  14.8%  [12%–17%]  22.3%  [20%–25%]  37.9%  [36%–40%]
                   Baseline : récence ÷ p90, × V         12.6%  [10%–16%]  19.1%  [17%–22%]  33.5%  [31%–36%]
                   Modèle total : P(total) × V              7.4%  [6%–9%]  12.2%  [10%–14%]  24.8%  [23%–27%]
                   Modèle partiel : P(partiel) × V       23.0%  [20%–25%]  30.9%  [28%–33%]  47.1%  [45%–49%]

In [42]:
# Tableau de décision : le modèle de chaque label, même K, même rétention, avec le repère « plus gros clients »
DU_RET, DU_MARGIN = 0.20, 0.2465                  # rétention : hypothèse du chiffrage d'équipe (non mesurée) ; marge brute
OWN = {"total": "Modèle total : P(total) × V", "part": "Modèle partiel : P(partiel) × V"}
SIZE = "Taille seule : V (plus gros clients)"
rows = []
for k in KS:
    res = {}
    for lab, nm in (("total", "Churn total"), ("part", "Churn partiel brut")):
        top = np.argpartition(-du_scores[OWN[lab]], k)[:k]
        top_v = np.argpartition(-du_scores[SIZE], k)[:k]
        y = du_te[f"y_{lab}"].to_numpy()
        ca, ca_v = LOST[lab][top].sum() / 1e6, LOST[lab][top_v].sum() / 1e6
        res[lab] = (ca, ca - ca_v)
        rows.append({"K": f"{k:,}", "Label": nm, "Précision du top-K": f"{y[top].mean():.0%}",
                     "Précision, plus gros clients": f"{y[top_v].mean():.0%}", "CA perdu capté (M)": f"{ca:,.0f}",
                     "dont surplus vs plus gros clients (M)": f"{ca - ca_v:,.0f}",
                     "Marge à 20 % : CA capté (M)": f"{ca * DU_RET * DU_MARGIN:,.1f}",
                     "Marge à 20 % : surplus (M)": f"{(ca - ca_v) * DU_RET * DU_MARGIN:,.1f}"})
    rows.append({"K": f"{k:,}", "Label": "→ rétention du partiel qui égale le total", "Précision du top-K": "",
                 "Précision, plus gros clients": "", "CA perdu capté (M)": "", "dont surplus vs plus gros clients (M)": "",
                 "Marge à 20 % : CA capté (M)": f"{DU_RET * res['total'][0] / res['part'][0]:.1%}",
                 "Marge à 20 % : surplus (M)": f"{DU_RET * res['total'][1] / res['part'][1]:.1%}"})
pd.DataFrame(rows).set_index(["K", "Label"])

Précision du top-K Précision, plus gros clients CA perdu capté (M) dont surplus vs plus gros clients (M) Marge à 20 % : CA capté (M)  \
K     Label                                                                                                                                                                            
1,000 Churn total                                              46%                           0%                 34                                    29                         1.7   
      Churn partiel brut                                       33%                          10%                116                                    41                         5.7   
      → rétention du partiel qui égale le total                                                                                                                                 5.9%   
2,000 Churn total                                              35%                           1%                 40                                    31                         2.0   
      Churn partiel brut                                       28%                          11%                156                                    43                         7.7   
      → rétention du partiel qui égale le total                                                                                                                                 5.1%   
5,000 Churn total                                              22%                           1%                 47                                    33                         2.3   
      Churn partiel brut                                       26%                          13%                238                                    46                        11.7   
      → rétention du partiel qui égale le total                                                                                                                                 3.9%   

                                                Marge à 20 % : surplus (M)  
K     Label                                                                 
1,000 Churn total                                                      1.4  
      Churn partiel brut                                               2.0  
      → rétention du partiel qui égale le total                      14.1%  
2,000 Churn total                                                      1.5  
      Churn partiel brut                                               2.1  
      → rétention du partiel qui égale le total                      14.4%  
5,000 Churn total                                                      1.6  
      Churn partiel brut                                               2.3  
      → rétention du partiel qui égale le total                      14.3%

**Lecture du test de label** (une seule photo de test, 22/03/2024 ; 61 286 clients réguliers ; montants dans le périmètre du dataset, annualisés). Chiffres cités pour K = 2 000, les conclusions sont les mêmes à 1 000 et à 5 000.

- **Le partiel est bien plus difficile à prédire, mais ça ne tranche pas.** ROC-AUC de 0,89 pour le total contre 0,68 pour le partiel ; le total est repéré avec un gain d'environ ×10 sur le taux de base (précision du top-K de 35 % pour un taux de base de 3,6 %), le partiel d'environ ×1,2 (28 % pour 23,5 %). L'hypothèse « le partiel est plus bruité » est donc confirmée. Mais **ce n'est pas une raison suffisante pour l'écarter** : le CA en jeu est 8 fois plus grand (505 M contre 61 M) et la liste du partiel capte beaucoup plus de CA.
- **CA perdu capté dans le top-2 000 : 40 M pour le total (65,7 %), 156 M pour le partiel (30,9 %)**, soit environ 4 fois plus. À rétention égale de 20 %, c'est 2,0 M contre 7,7 M de marge ; le partiel égale le total même avec une rétention de seulement 5 % (un quart de celle du total).
- **Cet écart surestime l'apport du modèle partiel.** Les plus gros clients perdent plus de CA en valeur absolue : la simple liste des 2 000 plus gros clients, sans modèle, capte déjà **22 % du CA du partiel** (contre 14,5 % pour le total). Face à ce repère, le surplus du modèle est de **43 M pour le partiel et 31 M pour le total**, soit 2,1 contre 1,5 M de marge : même ordre de grandeur. Le partiel égale alors le total avec une rétention de **14 %**, soit environ 70 % de celle du total.
- **La baseline sans apprentissage et le modèle total font mieux que « taille seule » sur le total, pas sur le partiel.** Pour le partiel, récence ÷ p90 × V (19 %) fait moins bien que la simple taille (22 %) : la récence ne signale pas une baisse de CA sans départ.
- **Les deux listes ne se recouvrent presque pas.** Le modèle total ne capte que 12 % du CA du partiel, le modèle partiel 26 % du CA du total : ce sont deux populations de clients à rappeler différentes.

**Pourquoi ces chiffres ne suffisent pas à choisir le partiel.**
1. **Il est brut.** 30 % des baisses de plus de 50 % ne durent pas (`04_churn_scope_total_vs_partial`) et le label « durable » ne se valide pas dans le temps : une partie des 156 M n'est pas du CA à sauver.
2. **Il est saisonnier.** Le taux de partiel varie de 10 à 14 % du CA selon la date de la photo ; un seul test ne permet pas de dire si l'avantage tient en toute saison (B/A = 0,96 sur cette photo).
3. **La rétention est inconnue.** Rattraper un client qui a baissé n'est pas la même action que regagner un client parti ; on ne peut pas supposer 20 % pour les deux, seul le pilote le mesurera.
4. **Une seule photo de test**, modèle par défaut, ρ non appliqué, même K pour les deux labels.

**Ce que le test change à la recommandation.** Le total reste le label le plus sûr (bien prédit, validable, rétention plus plausible). Mais le partiel n'est pas « une variante si le temps le permet » : c'est un second modèle crédible, surtout pour les gros clients, dont la rentabilité dépend d'une rétention à mesurer. Les deux utilisent le même pipeline ; un pilote randomisé sur les deux listes lèverait l'incertitude.

### Marge nette des trois réserves : de quoi dépend le choix ?

Le tableau de décision suppose que tout le CA capté est du CA sauvable, que la photo de test est représentative et que la rétention vaut 20 % pour les deux labels. On remplace ces trois hypothèses par des fourchettes :

1. **Persistance** (le partiel brut contient des baisses qui se résorbent seules, donc du CA qu'aucune action ne sauve). Part du CA du partiel qui **dure** (CA des 180 j suivants encore sous 50 % du CA de référence), **mesurée sur les 7 photos d'entraînement** (seules à avoir 360 j de recul), à partir des mêmes réguliers. Pour le churn total, 100 % par définition.
2. **Saison.** Le CA en jeu de chaque label varie selon la date de la photo ; facteur = CA en jeu de la photo / CA en jeu de la photo de test, sur les **13 photos mensuelles**. On suppose que la part captée par la liste reste constante.
3. **Rétention** (non mesurable ici) : 10 / 20 / 30 % pour le total, scénarios du chiffrage d'équipe. Pour le partiel on garde la même grille, et on calcule en plus **la rétention relative qu'il lui faut** pour égaler le total.

*Bas* = les trois réserves défavorables en même temps, *haut* = les trois favorables, *central* = médiane mesurée, saison de la photo de test, 20 % de rétention. Le CA de départ est celui du tableau précédent à K = 2 000, sous deux lectures : **CA capté** (toute la valeur de la liste) et **surplus** (ce que le modèle apporte en plus de la liste des plus gros clients, sans modèle). Marge brute 24,65 %, ρ non appliqué.

In [43]:
# Réserves 1 et 2, mesurées sur les mêmes réguliers : persistance des baisses et effet de la date de la photo
res_rows = []
for T in [str(t.date()) for t in pd.date_range("2023-03-22", "2024-03-22", freq=pd.DateOffset(months=1))]:
    p = du_te if T == DU_TEST else du_photo(T)
    r = {"Photo T": T, "CA en jeu, total (M)": p.lost_total.sum() / 1e6, "CA en jeu, partiel (M)": p.lost_part.sum() / 1e6, "persist": np.nan}
    if pd.Timestamp(T) + pd.Timedelta(days=2 * DU_H) <= DATA_END:                      # fenêtre C disponible
        r["persist"] = p.lost_part[p.rev_c < (1 - DROP) * p.rev_180].sum() / p.lost_part.sum()
    res_rows.append(r)
resv = pd.DataFrame(res_rows).set_index("Photo T")
resv["Facteur saison, total"] = resv["CA en jeu, total (M)"] / resv.loc[DU_TEST, "CA en jeu, total (M)"]
resv["Facteur saison, partiel"] = resv["CA en jeu, partiel (M)"] / resv.loc[DU_TEST, "CA en jeu, partiel (M)"]
shown = resv.rename(columns={"persist": "Part du partiel qui dure"})
shown.assign(**{c: shown[c].map("{:,.0f}".format) for c in shown.columns[:2]},
             **{"Part du partiel qui dure": shown["Part du partiel qui dure"].map(lambda v: "" if pd.isna(v) else f"{v:.0%}")},
             **{c: shown[c].map("{:.2f}".format) for c in shown.columns[3:]})

,"CA en jeu, total (M)","CA en jeu, partiel (M)",Part du partiel qui dure,"Facteur saison, total","Facteur saison, partiel"
Photo T,,,,,
2023-03-22,60,577,68%,0.99,1.14
2023-04-22,75,539,66%,1.23,1.07
2023-05-22,61,401,71%,1.00,0.79
2023-06-22,60,396,72%,0.99,0.78
2023-07-22,61,429,71%,1.00,0.85
2023-08-22,57,489,68%,0.94,0.97
2023-09-22,65,454,70%,1.08,0.90
2023-10-22,69,434,,1.13,0.86
2023-11-22,87,584,,1.43,1.16


In [44]:
# Marge annuelle à K = 2 000, en tenant compte des trois réserves (M€, périmètre du dataset)
KDEC = 2000
pers = resv.persist.dropna()
RV = {"total": {"persist": (1.0, 1.0, 1.0), "season": (resv["Facteur saison, total"].min(), 1.0, resv["Facteur saison, total"].max()),
                "ret": (0.10, 0.20, 0.30)},
      "part": {"persist": (pers.min(), pers.median(), pers.max()),
               "season": (resv["Facteur saison, partiel"].min(), 1.0, resv["Facteur saison, partiel"].max()), "ret": (0.10, 0.20, 0.30)}}
CA0 = {}                                                          # CA de départ à K = KDEC : capté, et surplus vs plus gros clients
top_v = np.argpartition(-du_scores[SIZE], KDEC)[:KDEC]
for lab in ("total", "part"):
    top = np.argpartition(-du_scores[OWN[lab]], KDEC)[:KDEC]
    CA0[lab] = {"CA capté": LOST[lab][top].sum() / 1e6, "Surplus vs plus gros clients": (LOST[lab][top].sum() - LOST[lab][top_v].sum()) / 1e6}


def marge(lab, base, **vary):
    """Marge (M) = CA de départ x persistance x saison x rétention x marge brute ; chaque réserve au centre sauf celle(s) imposée(s)."""
    idx = {"persist": 1, "season": 1, "ret": 1} | {k: v for k, v in vary.items()}
    f = 1.0
    for k in ("persist", "season", "ret"):
        f *= RV[lab][k][idx[k]]
    return CA0[lab][base] * f * DU_MARGIN


rows = []
for base in ("CA capté", "Surplus vs plus gros clients"):
    for lab, nm in (("total", "Churn total"), ("part", "Churn partiel brut")):
        rng_ = lambda k: f"{marge(lab, base, **{k: 0}):.1f} – {marge(lab, base, **{k: 2}):.1f}"
        rows.append({"Base": base, "Label": nm,
                     "Bas (3 réserves défavorables)": f"{marge(lab, base, persist=0, season=0, ret=0):.1f}",
                     "Central": f"{marge(lab, base):.1f}",
                     "Haut (3 réserves favorables)": f"{marge(lab, base, persist=2, season=2, ret=2):.1f}",
                     "Persistance seule": rng_("persist"), "Saison seule": rng_("season"), "Rétention seule": rng_("ret")})
marges = pd.DataFrame(rows).set_index(["Base", "Label"])
print(f"Persistance mesurée (part du CA du partiel qui dure, 7 photos) : min {pers.min():.0%} · médiane {pers.median():.0%} · max {pers.max():.0%}. "
      f"Facteurs de saison : total {RV['total']['season'][0]:.2f} à {RV['total']['season'][2]:.2f}, partiel {RV['part']['season'][0]:.2f} à {RV['part']['season'][2]:.2f}.")
marges

Persistance mesurée (part du CA du partiel qui dure, 7 photos) : min 66% · médiane 70% · max 72%. Facteurs de saison : total 0.94 à 1.43, partiel 0.78 à 1.22.


Bas (3 réserves défavorables) Central Haut (3 réserves favorables) Persistance seule Saison seule Rétention seule
Base                         Label                                                                                                                               
CA capté                     Churn total                                  0.9     2.0                          4.2         2.0 – 2.0    1.8 – 2.8       1.0 – 2.9
                             Churn partiel brut                           2.0     5.4                         10.1         5.1 – 5.5    4.2 – 6.5       2.7 – 8.0
Surplus vs plus gros clients Churn total                                  0.7     1.5                          3.3         1.5 – 1.5    1.4 – 2.2       0.8 – 2.3
                             Churn partiel brut                           0.6     1.5                          2.8         1.4 – 1.5    1.2 – 1.8       0.7 – 2.2

In [45]:
# Rétention relative que le partiel doit atteindre (r_partiel / r_total) pour égaler la marge du total
be = []
for base in ("CA capté", "Surplus vs plus gros clients"):
    num = CA0["total"][base] * RV["total"]["persist"][1] * RV["total"]["season"][1]
    row = {"Base": base}
    for nm, (pi, se) in {"Persistance basse": (0, 1), "Persistance centrale": (1, 1), "Persistance haute": (2, 1),
                         "Persistance et saison basses": (0, 0)}.items():
        row[nm] = f"{num / (CA0['part'][base] * RV['part']['persist'][pi] * RV['part']['season'][se]):.0%}"
    be.append(row)
print("Le partiel égale le total si sa rétention vaut au moins ce pourcentage de celle du total (saison du total au centre).")
pd.DataFrame(be).set_index("Base")

Le partiel égale le total si sa rétention vaut au moins ce pourcentage de celle du total (saison du total au centre).


,Persistance basse,Persistance centrale,Persistance haute,Persistance et saison basses
Base,,,,
CA capté,39%,37%,36%,49%
Surplus vs plus gros clients,108%,103%,100%,138%


**Lecture de la marge nette des réserves** (K = 2 000 ; M€ de marge par an, périmètre du dataset, non extrapolé au CA de 15,5 Md€).

- **Deux réserves sur trois se mesurent et pèsent peu.** La persistance du partiel est stable d'une photo à l'autre (66 à 72 %, médiane 70 %) : environ 30 % du CA du partiel brut n'est pas sauvable, c'est une décote fixe. La saison fait varier le CA en jeu de 0,94 à 1,43 pour le total (pics sur les photos de novembre-décembre) et de 0,78 à 1,22 pour le partiel : ±20 à 40 %, sans retourner le classement.
- **La rétention domine tout le reste.** Seule, elle fait varier la marge de 1,0 à 2,9 M (total) et de 2,7 à 8,0 M (partiel). C'est l'hypothèse non mesurée qui décide de l'ordre de grandeur, pas la saison ni la persistance.
- **Marge centrale** (persistance 70 %, saison de la photo de test, rétention 20 %). Lecture « CA capté » : **total 2,0 M** (0,9 à 4,2) contre **partiel 5,4 M** (2,0 à 10,1) ; le partiel reste devant le total central même dans son scénario bas. Lecture « surplus » face aux plus gros clients : **1,5 M contre 1,5 M**, donc à égalité (total 0,7 à 3,3 ; partiel 0,6 à 2,8).
- **Rétention relative nécessaire.** Pour égaler la marge du total, le partiel doit avoir une rétention d'au moins **37 %** de celle du total en lecture « CA capté », et d'environ **100 %** en lecture « surplus » (jusqu'à 138 % si persistance et saison sont défavorables).

**Ce qui tranche, et que les données ne disent pas.** Les deux lectures ne répondent pas à la même question : que font les commerciaux **aujourd'hui, sans modèle** ?
- S'ils ne suivent pas systématiquement les clients qui baissent (lecture « CA capté »), le partiel rapporte environ 2,5 fois plus que le total et ne perd que si sa rétention est inférieure à un peu plus du tiers de celle du total.
- S'ils appellent déjà les plus gros comptes (lecture « surplus »), les deux labels apportent la même chose, et le total est préférable : mieux prédit (AUC 0,89 contre 0,68), validable dans le temps, moins dépendant de la saison, avec une rétention plus plausible.

Les interviews disent que le churn est un « sujet chaud depuis des années » sans solution efficace, ce qui suggère l'absence de processus systématique, mais rien ne le prouve : c'est une question à poser à la Head of Sales. **Limites** : une seule photo de test ; la persistance est appliquée au surplus comme au CA capté, par simplification ; la saison suppose une part captée constante ; ρ n'est pas appliqué ; le recouvrement faible des deux listes laisse ouverte une liste mixte, non testée ici.

### Faut-il fusionner les deux labels ? Liste élargie et liste mixte

Le churn total est le cas limite du partiel (une baisse de 100 %) : on peut donc définir un label élargi, **« perte de CA ≥ 50 %, départ compris »**, et une seule liste des K plus gros perdants. On le compare aux deux listes séparées et à une **liste mixte** qui réserve une part de K au modèle total. Les listes sont comparées au même K (K = 2 000, modifiable via `KDEC`).

- **Élargi, un modèle** : un seul modèle entraîné sur le label élargi, score P(perte) × V.
- **Élargi, deux modèles** : on combine les deux modèles déjà entraînés, score (P(total) + c × P(partiel)) × V, où c est la perte moyenne d'un partiel en part de V (mesurée à l'entraînement).
- **Liste mixte** : les K₁ premiers du modèle total, complétés par les premiers du modèle partiel jusqu'à K clients distincts.

Les colonnes « déjà partis » et « baisse > 50 % » donnent la composition de la liste (un client ne compte que dans l'une des deux) ; « recouvrement » est la part de la liste présente aussi dans la liste du total ou du partiel seul.

In [46]:
# Label élargi : « perte de CA >= 50 %, départ compris » ; quatre façons de bâtir la liste de K clients
for d in (du_tr, du_te):
    d["y_any"] = ((d.y_total == 1) | (d.y_part == 1)).astype(int)
    d["lost_any"] = d.lost_total + d.lost_part
du_p["any"] = du_fit_predict("y_any")
c_part = du_tr.loc[du_tr.y_part == 1, "lost_part"].sum() / du_tr.loc[du_tr.y_part == 1, "V"].sum()   # perte moyenne d'un partiel, en part de V
LISTS = {"Total seul": du_p["total"] * V,
         "Partiel seul": du_p["part"] * V,
         "Élargi, un modèle : P(perte) × V": du_p["any"] * V,
         "Élargi, deux modèles : (P(total) + c × P(partiel)) × V": (du_p["total"] + c_part * du_p["part"]) * V}
tops = {n: np.argpartition(-s, KDEC)[:KDEC] for n, s in LISTS.items()}
yT, yP = du_te.y_total.to_numpy(), du_te.y_part.to_numpy()
LT, LP, LA = du_te.lost_total.to_numpy(), du_te.lost_part.to_numpy(), du_te.lost_any.to_numpy()
rows = []
for n, t in tops.items():
    rows.append({"Liste": n, "Déjà partis": f"{yT[t].mean():.0%}", "Baisse > 50 %": f"{yP[t].mean():.0%}",
                 "Aucun des deux": f"{1 - (yT[t] + yP[t]).mean():.0%}",
                 "CA du total capté": f"{LT[t].sum() / LT.sum():.0%}", "CA du partiel capté": f"{LP[t].sum() / LP.sum():.0%}",
                 "CA total + partiel capté (M)": f"{LA[t].sum() / 1e6:,.0f}",
                 "Recouvrement avec Total seul": f"{len(set(t) & set(tops['Total seul'])) / KDEC:.0%}",
                 "Recouvrement avec Partiel seul": f"{len(set(t) & set(tops['Partiel seul'])) / KDEC:.0%}"})
print(f"Test du {DU_TEST}, K = {KDEC:,}. CA en jeu : total {LT.sum() / 1e6:,.0f} M, partiel {LP.sum() / 1e6:,.0f} M, ensemble {LA.sum() / 1e6:,.0f} M "
      f"(le partiel pèse {LP.sum() / LA.sum():.0%}). Perte moyenne d'un partiel : c = {c_part:.0%} de V.")
pd.DataFrame(rows).set_index("Liste")

Test du 2024-03-22, K = 2,000. CA en jeu : total 61 M, partiel 505 M, ensemble 566 M (le partiel pèse 89%). Perte moyenne d'un partiel : c = 71% de V.


,Déjà partis,Baisse > 50 %,Aucun des deux,CA du total capté,CA du partiel capté,CA total + partiel capté (M),Recouvrement avec Total seul,Recouvrement avec Partiel seul
Liste,,,,,,,,
Total seul,35%,36%,29%,66%,12%,102,100%,16%
Partiel seul,3%,28%,69%,26%,31%,172,16%,100%
"Élargi, un modèle : P(perte) × V",7%,28%,65%,43%,29%,175,22%,85%
"Élargi, deux modèles : (P(total) + c × P(partiel)) × V",9%,29%,63%,45%,30%,180,26%,90%


In [47]:
# Liste mixte : K1 clients du modèle total, le reste du modèle partiel (doublons retirés)
def liste_mixte(k_total: int, K: int = KDEC) -> np.ndarray:
    pick = list(np.argsort(-LISTS["Total seul"])[:k_total])
    seen = set(pick)
    for i in np.argsort(-LISTS["Partiel seul"]):
        if len(pick) >= K:
            break
        if i not in seen:
            pick.append(i)
            seen.add(i)
    return np.array(pick)


rows = []
for kt in (0, 250, 500, 1000, 1500, KDEC):
    t = liste_mixte(kt)
    rows.append({"Places réservées au total": f"{kt:,}", "Places partiel": f"{KDEC - kt:,}",
                 "CA du total capté": f"{LT[t].sum() / LT.sum():.0%}", "CA du partiel capté": f"{LP[t].sum() / LP.sum():.0%}",
                 "CA total + partiel capté (M)": f"{LA[t].sum() / 1e6:,.0f}", "Part de vrais cas dans la liste": f"{(yT[t] + yP[t]).mean():.0%}"})
pd.DataFrame(rows).set_index("Places réservées au total")

,Places partiel,CA du total capté,CA du partiel capté,CA total + partiel capté (M),Part de vrais cas dans la liste
Places réservées au total,,,,,
0,"2,000",26%,31%,172,31%
250,"1,750",44%,30%,179,37%
500,"1,500",51%,29%,179,44%
"1,000","1,000",59%,26%,169,54%
"1,500",500,64%,23%,153,64%
"2,000",0,66%,12%,102,71%


**Lecture** (K = 2 000, une seule photo de test ; pas d'intervalle de confiance calculé sur ces comparaisons de listes, donc les écarts de quelques millions sont à lire comme des égalités).

- **Le label élargi se comporte comme le partiel.** Le partiel pèse 89 % du CA en jeu de l'ensemble (505 M sur 566 M) : la liste élargie recouvre à 85-90 % celle du partiel seul, et seulement à 22-26 % celle du total.
- **Il dilue le total sans rien apporter de plus au partiel.** Il capte 43-45 % du CA du total, contre 66 % pour la liste dédiée, et 29-30 % du CA du partiel, contre 31 % pour la liste dédiée. Au total, 175-180 M captés contre 172 M pour le partiel seul : un gain de 2 à 5 %, dans le bruit.
- **La liste du total est celle qui se trompe le moins.** 71 % de ses clients ont déjà quitté ou perdu plus de la moitié de leur CA, contre 31 % pour la liste du partiel et 35-37 % pour la liste élargie. Elle rapporte moins de CA mais fait perdre moins de temps aux commerciaux, ce qui compte pour l'adoption.
- **Une liste mixte vaut la liste élargie, sans perdre le total.** Avec 250 à 500 places réservées au total, on capte 179 M (comme la liste élargie) tout en gardant 44 à 51 % du CA du total ; la part de vrais cas dans la liste passe de 31 % à 37-44 %. Réserver 1 000 places ou plus coûte du CA du partiel sans gain équivalent.

**Conséquence.** On n'a pas à choisir un seul label pour le modèle : on peut livrer deux scores et une règle de répartition des K clients. Ce qui reste à décider n'est pas le label mais la **répartition de la capacité** entre les deux listes, la valeur de rétention de chacune (non mesurée, à lire dans la marge ci-dessus) et le chiffre que la présentation mettra en avant.

### Répartition de K entre le modèle total et le modèle partiel, à K = 1 000 et K = 4 000

La capacité supposée (100 commerciaux × 5 appels par semaine ≈ 2 000 appels par mois pour l'entreprise) correspond, à l'échelle du dataset (≈ 30 % du CA), à **≈ 600 à 760 appels par mois**, soit **K ≈ 1 000** pour une liste mensuelle, et à **≈ 3 600 à 4 560 clients distincts** sur 180 j si six listes s'accumulent sans rappel, soit **K ≈ 4 000**. Pour chaque K, on réserve une part des places au modèle total (le reste au partiel, doublons remplacés) et on mesure le CA capté.

Comme les écarts entre répartitions sont petits, on ajoute un **tirage bootstrap sur les clients** (200 tirages) : pour chaque lecture, on compte la part des tirages où chaque répartition est la meilleure. Les marges utilisent la rétention de 20 % pour le total ; pour le partiel, soit la même (20 %), soit la moitié (10 %). Deux lectures du CA : **CA capté** (aucune action aujourd'hui) ou **surplus** face à la liste des K plus gros clients (déjà suivis).

In [48]:
# Répartition de K : composition et CA capté pour chaque part de places réservée au total
K_TEST = [1000, 4000]
FRACS = [0.0, 0.10, 0.25, 0.50, 0.75, 1.0]                  # part des K places réservée au modèle total
ST, SP = LISTS["Total seul"], LISTS["Partiel seul"]
N_TE = len(du_te)


def pick_mixed(order_t, order_p, kt, K):
    """kt premiers du modèle total, puis les premiers du modèle partiel non déjà pris, jusqu'à K clients."""
    sel = np.zeros(N_TE, bool)
    sel[order_t[:kt]] = True
    free = ~sel[order_p]
    return np.concatenate([order_t[:kt], order_p[free & (np.cumsum(free) <= K - kt)]])


OT, OP = np.argsort(-ST), np.argsort(-SP)
rows = []
for K in K_TEST:
    for f in FRACS:
        t = pick_mixed(OT, OP, int(f * K), K)
        rows.append({"K": f"{K:,}", "Places réservées au total": f"{int(f * K):,} ({f:.0%})",
                     "CA du total capté": f"{LT[t].sum() / LT.sum():.0%}", "CA du partiel capté": f"{LP[t].sum() / LP.sum():.0%}",
                     "Total + partiel capté (M)": f"{LA[t].sum() / 1e6:,.0f}", "Part de vrais cas": f"{(yT[t] + yP[t]).mean():.0%}"})
pd.DataFrame(rows).set_index(["K", "Places réservées au total"])

CA du total capté CA du partiel capté Total + partiel capté (M) Part de vrais cas
K     Places réservées au total                                                                                  
1,000 0 (0%)                                  18%                 23%                       127               36%
      100 (10%)                               32%                 22%                       132               40%
      250 (25%)                               40%                 21%                       133               47%
      500 (50%)                               47%                 19%                       125               59%
      750 (75%)                               52%                 16%                       113               71%
      1,000 (100%)                            56%                  7%                        71               79%
4,000 0 (0%)                                  36%                 42%                       235               29%
      400 (10%)                               54%                 42%                       244               33%
      1,000 (25%)                             63%                 40%                       242               39%
      2,000 (50%)                             69%                 37%                       228               48%
      3,000 (75%)                             73%                 32%                       208               56%
      4,000 (100%)                            74%                 20%                       147               61%

In [49]:
# Quelle répartition maximise la marge ? Part des 200 tirages bootstrap où chaque répartition est la meilleure
SCN = {"CA capté, r partiel = r total": (1.0, False), "CA capté, r partiel = r total / 2": (0.5, False),
       "Surplus, r partiel = r total": (1.0, True), "Surplus, r partiel = r total / 2": (0.5, True)}
unit = DU_RET * DU_MARGIN / 1e6                              # CA annualisé -> marge, en M€


def margins(idx, K):
    """Marge de chaque répartition (lignes) sous chaque scénario (colonnes), sur l'échantillon idx de clients."""
    ot, op = np.argsort(-ST[idx]), np.argsort(-SP[idx])
    lt, lp = LT[idx], LP[idx]
    top_v = np.argpartition(-V[idx], K)[:K]
    out = np.zeros((len(FRACS), len(SCN)))
    for a, f in enumerate(FRACS):
        sel = np.zeros(len(idx), bool)
        sel[ot[:int(f * K)]] = True
        free = ~sel[op]
        t = np.concatenate([ot[:int(f * K)], op[free & (np.cumsum(free) <= K - int(f * K))]])
        for b, (w, surplus) in enumerate(SCN.values()):
            base_t, base_p = (lt[top_v].sum(), lp[top_v].sum()) if surplus else (0.0, 0.0)
            out[a, b] = unit * ((lt[t].sum() - base_t) + w * (lp[t].sum() - base_p))
    return out


boot = np.random.default_rng(1)
tirages = [boot.integers(0, N_TE, N_TE) for _ in range(200)]
tables = {}
for K in K_TEST:
    point = margins(np.arange(N_TE), K)
    runs = np.stack([margins(i, K) for i in tirages])        # (tirages, répartitions, scénarios)
    best = np.argmax(runs, axis=1)                           # meilleure répartition de chaque tirage
    rows = []
    for b, name in enumerate(SCN):
        r = {"K": f"{K:,}", "Scénario": name}
        for a, f in enumerate(FRACS):
            r[f"{f:.0%} au total"] = f"{point[a, b]:.1f}  ({(best[:, b] == a).mean():.0%})"
        rows.append(r)
    tables[K] = pd.DataFrame(rows)
print("Marge annuelle en M€ (périmètre du dataset) ; entre parenthèses, part des tirages où cette répartition est la meilleure.")
pd.concat(tables.values()).set_index(["K", "Scénario"])

Marge annuelle en M€ (périmètre du dataset) ; entre parenthèses, part des tirages où cette répartition est la meilleure.


0% au total 10% au total 25% au total 50% au total 75% au total 100% au total
K     Scénario                                                                                                       
1,000 CA capté, r partiel = r total       6.3  (0%)   6.5  (46%)   6.5  (55%)    6.1  (0%)    5.6  (0%)     3.5  (0%)
      CA capté, r partiel = r total / 2   3.4  (0%)    3.7  (0%)   3.9  (88%)   3.8  (12%)    3.6  (0%)     2.6  (0%)
      Surplus, r partiel = r total        2.3  (0%)   2.6  (46%)   2.6  (55%)    2.2  (0%)    1.7  (0%)    -0.4  (0%)
      Surplus, r partiel = r total / 2    1.3  (0%)    1.6  (0%)   1.8  (88%)   1.7  (12%)    1.5  (0%)     0.5  (0%)
4,000 CA capté, r partiel = r total      11.6  (0%)  12.0  (91%)   11.9  (9%)   11.3  (0%)   10.2  (0%)     7.2  (0%)
      CA capté, r partiel = r total / 2   6.3  (0%)    6.8  (2%)   6.9  (98%)    6.7  (0%)    6.2  (0%)     4.7  (0%)
      Surplus, r partiel = r total        2.6  (0%)   3.1  (91%)    3.0  (9%)    2.3  (0%)    1.3  (0%)    -1.7  (0%)
      Surplus, r partiel = r total / 2    1.5  (0%)    2.0  (2%)   2.1  (98%)    1.9  (0%)    1.4  (0%)    -0.0  (0%)

**Lecture de la répartition** (photo de test du 22/03/2024 ; les parts entre parenthèses viennent d'un tirage bootstrap sur les clients d'une seule photo : elles mesurent le bruit d'échantillonnage, pas la variation d'une saison à l'autre).

- **Conclusion robuste : réserver 10 à 25 % des places au modèle total.** Sur les 8 combinaisons (2 valeurs de K × 4 scénarios), la meilleure répartition est toujours dans cette fourchette. Donner 0 % au total n'est jamais la meilleure option, et 100 % est la pire dans tous les cas.
- **Le gain face à une liste 100 % partiel est faible à rétention égale (+3 %), mais devient net si la rétention du partiel est plus faible ou si les commerciaux suivent déjà les plus gros comptes** : +10 à +15 % à rétention moitié (K = 1 000 : 3,4 → 3,9 M ; K = 4 000 : 6,3 → 6,9 M), +13 à +19 % en lecture « surplus » (2,3 → 2,6 M ; 2,6 → 3,1 M), jusqu'à +40 % en cumulant les deux (1,3 → 1,8 M ; 1,5 → 2,1 M). Le quota du total est donc une **assurance contre l'incertitude sur la rétention**, pas un gain de CA assuré.
- **10 % ou 25 % ?** Les tirages ne les départagent pas à K = 1 000 (46 % contre 55 %) et se partagent selon le scénario à K = 4 000. **25 % ne perd jamais plus de 0,1 M€ face à la meilleure répartition** et gagne jusqu'à 0,2 M€ dans les scénarios prudents : c'est le choix le moins risqué, soit **≈ 250 places à K = 1 000 et ≈ 1 000 à K = 4 000**.
- **Coût pour le chiffre de tête (le total).** Avec 25 % des places, le total capte 40 % (K = 1 000) ou 63 % (K = 4 000) de son CA en jeu, contre 56 % ou 74 % si toute la liste lui était réservée : environ 1,2 M€ de marge au lieu de 1,7 M€, et 1,9 M€ au lieu de 2,2 M€. Le partiel apporte le complément (≈ 5 M€ à K = 1 000 et ≈ 10 M€ à K = 4 000 à rétention égale, moitié moins à rétention moitié).
- **Qualité de la liste.** La part de vrais cas passe de 36 % (100 % partiel) à 47 % avec 25 % de places au total à K = 1 000, et de 29 % à 39 % à K = 4 000 : moins d'appels inutiles pour les commerciaux.

*Limites.* Une seule photo de test ; les scénarios de rétention (égale ou moitié) sont des hypothèses, et la valeur absolue des marges dépend de la rétention de 20 % ; marges sur le périmètre du dataset, non extrapolées.

#### Scénario à 1 000 appels par semaine

1 000 appels par semaine pour l'entreprise font environ 4 000 par mois ; à l'échelle du dataset (≈ 30 à 38 % de l'entreprise), **≈ 1 200 à 1 500 appels par mois**, soit **K ≈ 1 400** pour une liste mensuelle, et **≈ 7 200 à 9 100 clients distincts** sur 180 j avec six listes sans rappel, soit **K ≈ 8 000**. Mêmes méthode, scénarios et tirages que ci-dessus, avec ces deux valeurs de K.

In [50]:
# Scénario 1 000 appels par semaine : composition et CA capté selon la part de places réservée au total
K_TEST_B = [1400, 8000]
rows = []
for K in K_TEST_B:
    for f in FRACS:
        t = pick_mixed(OT, OP, int(f * K), K)
        rows.append({"K": f"{K:,}", "Places réservées au total": f"{int(f * K):,} ({f:.0%})",
                     "CA du total capté": f"{LT[t].sum() / LT.sum():.0%}", "CA du partiel capté": f"{LP[t].sum() / LP.sum():.0%}",
                     "Total + partiel capté (M)": f"{LA[t].sum() / 1e6:,.0f}", "Part de vrais cas": f"{(yT[t] + yP[t]).mean():.0%}"})
pd.DataFrame(rows).set_index(["K", "Places réservées au total"])

CA du total capté CA du partiel capté Total + partiel capté (M) Part de vrais cas
K     Places réservées au total                                                                                  
1,400 0 (0%)                                  21%                 26%                       146               33%
      140 (10%)                               37%                 26%                       153               38%
      350 (25%)                               45%                 25%                       153               45%
      700 (50%)                               54%                 23%                       147               57%
      1,050 (75%)                             58%                 19%                       132               67%
      1,400 (100%)                            61%                  9%                        81               75%
8,000 0 (0%)                                  47%                 56%                       313               27%
      800 (10%)                               66%                 56%                       322               31%
      2,000 (25%)                             73%                 55%                       321               36%
      4,000 (50%)                             78%                 52%                       309               42%
      6,000 (75%)                             81%                 47%                       285               48%
      8,000 (100%)                            81%                 35%                       227               53%

In [51]:
# Scénario 1 000 appels par semaine : marge et part des 200 tirages où chaque répartition est la meilleure
tab_b = []
for K in K_TEST_B:
    point = margins(np.arange(N_TE), K)
    best = np.argmax(np.stack([margins(i, K) for i in tirages]), axis=1)
    for b, name in enumerate(SCN):
        r = {"K": f"{K:,}", "Scénario": name}
        for a, f in enumerate(FRACS):
            r[f"{f:.0%} au total"] = f"{point[a, b]:.1f}  ({(best[:, b] == a).mean():.0%})"
        tab_b.append(r)
print("Marge annuelle en M€ (périmètre du dataset) ; entre parenthèses, part des tirages où cette répartition est la meilleure.")
pd.DataFrame(tab_b).set_index(["K", "Scénario"])

Marge annuelle en M€ (périmètre du dataset) ; entre parenthèses, part des tirages où cette répartition est la meilleure.


0% au total 10% au total 25% au total 50% au total 75% au total 100% au total
K     Scénario                                                                                                       
1,400 CA capté, r partiel = r total       7.2  (0%)   7.5  (55%)   7.5  (45%)    7.3  (0%)    6.5  (0%)     4.0  (0%)
      CA capté, r partiel = r total / 2   3.9  (0%)    4.3  (0%)   4.4  (55%)   4.4  (44%)    4.1  (0%)     2.9  (0%)
      Surplus, r partiel = r total        2.3  (0%)   2.6  (55%)   2.6  (45%)    2.3  (0%)    1.6  (0%)    -0.9  (0%)
      Surplus, r partiel = r total / 2    1.3  (0%)    1.7  (0%)   1.8  (55%)   1.8  (44%)    1.5  (0%)     0.3  (0%)
8,000 CA capté, r partiel = r total      15.4  (0%)  15.9  (94%)   15.8  (6%)   15.2  (0%)   14.0  (0%)    11.2  (0%)
      CA capté, r partiel = r total / 2   8.4  (0%)    8.9  (2%)   9.0  (98%)    8.8  (0%)    8.2  (0%)     6.8  (0%)
      Surplus, r partiel = r total        2.3  (0%)   2.7  (94%)    2.7  (6%)    2.1  (0%)    0.9  (0%)    -2.0  (0%)
      Surplus, r partiel = r total / 2    1.3  (0%)    1.8  (2%)   1.9  (98%)    1.7  (0%)    1.1  (0%)    -0.3  (0%)

**Lecture du scénario à 1 000 appels par semaine** (même photo de test, mêmes réserves que ci-dessus).

- **Même conclusion : réserver 10 à 25 % des places au total.** À K = 1 400 comme à K = 8 000, 0 % au total n'est jamais la meilleure répartition et 100 % est la pire. À K = 1 400, quand la rétention du partiel est moitié moindre, 25 % et 50 % sont à égalité (44 à 55 % des tirages chacun) : la fourchette utile s'élargit un peu vers le haut. 25 % ne perd jamais plus de 0,1 M€ face à la meilleure répartition dans les 8 scénarios.
- **Gain du quota** (25 % contre 0 %) : +3 à +4 % à rétention égale (7,2 → 7,5 M ; 15,4 → 15,8 M), +7 à +13 % à rétention moitié (3,9 → 4,4 M ; 8,4 → 9,0 M), +13 à +17 % en lecture « surplus » (2,3 → 2,6 M ; 2,3 → 2,7 M), jusqu'à +46 % en cumulant (1,3 → 1,9 M).
- **Rendements décroissants.** Doubler la capacité (500 → 1 000 appels par semaine) ne double pas la marge : en lecture « CA capté » à rétention égale, elle passe de 6,5 à 7,5 M€ pour la liste mensuelle (K = 1 000 → 1 400) et de 12,0 à 15,9 M€ pour la liste cumulée (K = 4 000 → 8 000), soit +15 % et +33 %. En lecture « surplus », elle ne bouge presque pas (2,6 → 2,6 M ; 3,1 → 2,7 M) : face à une liste des plus gros clients, élargir la liste apporte peu.
- **Le total captant déjà l'essentiel.** À K = 8 000, la liste 100 % total capte 81 % de son CA en jeu (73 % avec 25 % des places) : ajouter des appels au total n'apporte presque plus rien ; c'est le partiel qui absorbe la capacité supplémentaire (56 % de son CA en jeu à 8 000, contre 25 % à 1 400).
- **Coût pour le chiffre de tête** avec 25 % des places : le total capte 45 % (K = 1 400) ou 73 % (K = 8 000) de son CA en jeu, contre 61 % et 81 % avec toute la liste.

## Pour la suite

Le label d'entraînement (180 jours pour tous, ou relatif au rythme) est calibré et comparé dans `06_label_horizon_calibration` et `08_label_rolling_backtest` ; la décote du churn total est mesurée dans `07_realized_revenue_loss`. Les décisions finales sont dans `docs/modelisation.md`.